In [ ]:
import sys
import random
import threading
import queue
import time
import carla
import cv2
import numpy as np
import torch
from ultralytics import YOLO
import warnings
from datetime import datetime
from torchvision import transforms
from collections import deque
import os
from sklearn.linear_model import RANSACRegressor

# Suppress warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

class PIDController:
    def __init__(self, Kp=0.0, Ki=0.0, Kd=0.0):
        self.Kp = Kp
        self.Ki = Ki
        self.Kd = Kd
        self.prev_error = 0
        self.integral = 0
        
    def update(self, error, dt=1.0):
        self.integral += error * dt
        derivative = (error - self.prev_error) / dt
        output = self.Kp * error + self.Ki * self.integral + self.Kd * derivative
        self.prev_error = error
        return output

class AutonomousVehicle:
    def __init__(self, port=2000):
        """Initialize all systems with safety parameters"""
        # CARLA connection
        self.client = carla.Client('localhost', port)
        self.client.set_timeout(20.0)
        self.world = self.client.get_world()
        self.map = self.world.get_map()
        
        # System components
        self.actors = []
        self.running = False
        self.image_queue = queue.Queue(maxsize=1)
        self.vehicle = None
        self.camera = None
        self.collision_sensor = None
        self.current_waypoint = None
        self.tracked_obstacles = {}
        self.lane_boundaries = None
        self.depth_model = None
        self.depth_transform = None
        self.frame_count = 0
        self.global_route = []
        self.current_goal = None
        self.local_trajectory = []
        self.avoidance_mode = False
        self.safety_thread = threading.Thread(target=self._safety_monitor, daemon=True)
        self.safety_status = "NORMAL"
        self.min_safe_distance = 5.0
        self.pid_steer = PIDController(Kp=0.8, Ki=0.001, Kd=0.1)
        self.pid_speed = PIDController(Kp=0.5, Ki=0.001, Kd=0.01)
        self.last_control_time = time.time()
        self.last_recovery_time = 0
        self.recovery_cooldown = 5.0
        self.lane_offset_history = deque(maxlen=5)
        self.steering_history = deque(maxlen=3)
        self.lane_departure_warning = False
        self.lane_change_in_progress = False

        # Detection colors
        self.detection_colors = {
            'close': (0, 0, 255),    # Red
            'medium': (0, 165, 255), # Orange
            'far': (0, 255, 0)      # Green
        }
        self.route_color = (255, 0, 0)
        self.route_thickness = 2

        # Reverse parameters
        self.reverse_mode = False
        self.reverse_start_time = 0
        self.reverse_duration = 2.0
        self.min_reverse_distance = 3.0

        # Perspective transform for lane detection
        self.perspective_transform = None
        self.inverse_perspective = None
        self._init_perspective_transform()

        # Perception system
        self.model = self._initialize_perception()
        self._initialize_depth_estimation()
        
        # Control parameters
        self.target_speed = 40  # km/h
        self.emergency_brake = False
        self.off_road_counter = 0
        self.max_off_road_frames = 15
        self.current_obstacles = []
        self.last_collision_time = 0
        self.collision_cooldown = 2.0
        self.lane_keeping_active = True
        self.static_obstacle_range = 15.0
        self.lane_change_threshold = 1.5  # meters
        
        # Visualization
        self.log_file = open('vehicle7.csv', 'w')
        self.log_file.write("timestamp,speed,throttle,steer,brake,obstacles,road_status,collision,lane_offset\n")

    def _init_perspective_transform(self):
        """Improved perspective transform calibration"""
        # These values should be calibrated for your camera setup
        src = np.float32([
            [200, 720],  # Bottom-left
            [560, 470],  # Top-left (adjusted for better coverage)
            [720, 470],  # Top-right (adjusted for better coverage)
            [1100, 720]  # Bottom-right
        ])
        
        dst = np.float32([
            [320, 720],  # Bottom-left (wider view)
            [320, 0],    # Top-left
            [960, 0],    # Top-right
            [960, 720]   # Bottom-right
        ])
        
        self.perspective_transform = cv2.getPerspectiveTransform(src, dst)
        self.inverse_perspective = cv2.getPerspectiveTransform(dst, src)

    def _initialize_perception(self):
        """Initialize YOLOv8 model with optimized settings"""
        try:
            model = YOLO('yolov8m.pt').to('cpu')
            model.classes = [0, 1, 2, 3, 5, 7, 8]  # Vehicles, pedestrians, bikes, traffic signs
            model.conf = 0.7
            print("🟢 Perception system initialized")
            return model
        except Exception as e:
            print(f"🔴 Failed to initialize perception: {str(e)}")
            raise

    def _initialize_depth_estimation(self):
        """Initialize MiDaS monocular depth estimation"""
        try:
            torch.hub.set_dir('/tmp/torch_hub')
            self.depth_model = torch.hub.load(
                'intel-isl/MiDaS', 
                'MiDaS_small',
                verbose=False
            ).to('cpu')
            
            # Warm-up the model
            with torch.no_grad():
                dummy_input = torch.rand(1, 3, 256, 256).to('cpu')
                _ = self.depth_model(dummy_input)
            
            self.depth_transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                   std=[0.229, 0.224, 0.225]),
                lambda x: x.unsqueeze(0)
            ])
            print("✅ Depth estimation initialized")
            return True
        except Exception as e:
            print(f"❌ Depth initialization failed: {str(e)}")
            self.depth_model = None
            return False

    def _process_depth(self, frame):
        """Convert RGB frame to depth map with size validation"""
        if self.depth_model is None or frame is None:
            return None
        
        try:
            # Resize to compatible dimensions (multiples of 32)
            h, w = frame.shape[:2]
            new_h = (h // 32) * 32
            new_w = (w // 32) * 32
            resized_frame = cv2.resize(frame, (new_w, new_h))
            
            input_tensor = self.depth_transform(resized_frame).to('cpu')
            with torch.no_grad():
                prediction = self.depth_model(input_tensor)
            
            depth_map = torch.nn.functional.interpolate(
                prediction.unsqueeze(1),
                size=(h, w),  # Resize back to original
                mode="bicubic",
                align_corners=False
            ).squeeze()
            
            # Convert to meters and normalize
            depth_map = (depth_map - depth_map.min()) * (25.0 / (depth_map.max() - depth_map.min()))
            
            # Apply Gaussian blur for smoother output
            depth_map = cv2.GaussianBlur(depth_map.cpu().numpy(), (5, 5), 0)
            
            return depth_map
        except Exception as e:
            print(f"⚠️ Depth processing error: {str(e)}")
            return None

    def _get_obstacle_distance(self, depth_map, x, y):
        """Get median depth in a small region around (x,y)"""
        try:
            patch = depth_map[
                max(0,y-2):min(y+3,depth_map.shape[0]),
                max(0,x-2):min(x+3,depth_map.shape[1])
            ]
            return np.median(patch)
        except:
            return 20.0  # Safe default

    def cleanup(self):
        """Clean up all resources"""
        print("🧹 Cleaning up resources...")
        self.running = False
        
        try:
            if self.safety_thread.is_alive():
                self.safety_thread.join(timeout=1.0)
        except:
            pass
            
        for actor in self.actors:
            try:
                if actor.is_alive:
                    actor.destroy()
            except:
                continue
                
        self.actors = []
        self.log_file.close()
        
        try:
            cv2.destroyAllWindows()
        except:
            pass
            
        print("🟢 Cleanup complete")

    def spawn_vehicle(self):
        """Enhanced vehicle spawning with better obstacle checks"""
        self._destroy_existing_vehicle()
    
        # Get all spawn points and filter for safety
        spawn_points = self.map.get_spawn_points()
        safe_spawns = []
    
        for point in spawn_points:
            wp = self.map.get_waypoint(point.location)
            if wp and wp.lane_type == carla.LaneType.Driving:
                # Check for nearby vehicles
                nearby_vehicles = [a for a in self.world.get_actors().filter('vehicle.*') 
                                if a.get_location().distance(point.location) < 15.0]
                if not nearby_vehicles:
                    safe_spawns.append(point)
    
        # Fallback to any drivable spawn point if no safe ones found
        if not safe_spawns:
            print("⚠️ No optimal spawn points found! Using fallback")
            safe_spawns = [sp for sp in spawn_points 
                        if self.map.get_waypoint(sp.location).lane_type == carla.LaneType.Driving]
    
        # Try spawning with multiple attempts
        for attempt in range(3):
            spawn_point = random.choice(safe_spawns) if safe_spawns else random.choice(spawn_points)
            blueprint = self._select_vehicle_blueprint()
            try:
                self.vehicle = self.world.spawn_actor(blueprint, spawn_point)
                if self.vehicle:
                    self.actors.append(self.vehicle)
                    self.current_waypoint = self.map.get_waypoint(spawn_point.location)
                    print(f"🚗 Spawned {self.vehicle.type_id} at safe location")
                    return True
            except Exception as e:
                print(f"⚠️ Spawn attempt {attempt+1} failed: {str(e)}")
                time.sleep(1)
    
        return False    
    def _destroy_existing_vehicle(self):
        """Cleanup previous vehicle"""
        if self.vehicle and self.vehicle.is_alive:
            try:
                self.vehicle.set_autopilot(False)
                self.vehicle.set_simulate_physics(False)
                time.sleep(0.2)
                self.vehicle.destroy()
                time.sleep(0.5)
            except:
                pass

    def _select_vehicle_blueprint(self):
        """Choose appropriate vehicle model with proper attribute handling"""
        # Get all vehicle blueprints excluding motorcycles and bicycles
        blueprints = [bp for bp in self.world.get_blueprint_library().filter('vehicle.*') 
                    if 'motorcycle' not in bp.id and 'bicycle' not in bp.id]
    
        # Select a random blueprint
        blueprint = random.choice(blueprints)
    
        # Set basic attributes
        blueprint.set_attribute('color', '255,0,0')  # Red color
        blueprint.set_attribute('role_name', 'ego_vehicle')
    
        # Don't try to set non-existent attributes
        return blueprint

    def setup_sensors(self):
        """Initialize all sensors with proper error handling and dependency management"""
        # First ensure we have a vehicle
        if not self.vehicle:
            print("🔴 Cannot setup sensors - no vehicle available")
            return False
    
        # Setup camera (essential)
        if not self._setup_camera():
            print("🔴 Aborting sensor setup - camera failed")
            return False
    
        # Setup collision sensor (important but not critical)
        collision_sensor_ok = self._setup_collision_sensor()
        if not collision_sensor_ok:
            print("⚠️ Proceeding without collision sensor - some safety features may be limited")
    
        # Setup lane invasion sensor (optional)
        lane_sensor_ok = self._setup_lane_invasion_sensor()
        if not lane_sensor_ok:
            print("⚠️ Proceeding without lane invasion sensor")
    
        # Consider setup successful if we at least have the camera
        return True
    def _setup_camera(self):
        """Configure RGB camera with better parameters"""
        try:
            blueprint = self.world.get_blueprint_library().find('sensor.camera.rgb')
            blueprint.set_attribute('image_size_x', '1280')
            blueprint.set_attribute('image_size_y', '720')
            blueprint.set_attribute('fov', '90')
            blueprint.set_attribute('motion_blur_intensity', '0.5')  # More realistic
            
            self.camera = self.world.spawn_actor(
                blueprint,
                carla.Transform(carla.Location(x=1.6, z=1.7), carla.Rotation(pitch=-5)),
                attach_to=self.vehicle
            )
            self.camera.listen(self._camera_callback)
            self.actors.append(self.camera)
            print("📷 Camera sensor ready")
            return True
        except Exception as e:
            print(f"🔴 Camera failed: {str(e)}")
            return False

    def _setup_collision_sensor(self):
        """Initialize collision detector with proper error handling"""
        try:
            blueprint = self.world.get_blueprint_library().find('sensor.other.collision')
        
            # Remove attempt to set non-existent attribute
            # blueprint.set_attribute('noise_stddev', '0.1')  # This line was causing the error
        
            self.collision_sensor = self.world.spawn_actor(
                blueprint,
                carla.Transform(),
                attach_to=self.vehicle
            )
            self.collision_sensor.listen(self._on_collision)
            self.actors.append(self.collision_sensor)
            print("⚠️ Collision sensor ready")
            return True
        except Exception as e:
            print(f"🔴 Collision sensor failed: {str(e)}")
            return False

    def _setup_lane_invasion_sensor(self):
        """Initialize lane invasion detector"""
        try:
            blueprint = self.world.get_blueprint_library().find('sensor.other.lane_invasion')
            lane_sensor = self.world.spawn_actor(
                blueprint,
                carla.Transform(),
                attach_to=self.vehicle
            )
            lane_sensor.listen(self._on_lane_invasion)
            self.actors.append(lane_sensor)
            print("🛣️ Lane invasion sensor ready")
            return True
        except Exception as e:
            print(f"🔴 Lane sensor failed: {str(e)}")
            return False

    def _camera_callback(self, image):
        """Process camera images with better error handling"""
        try:
            array = np.frombuffer(image.raw_data, dtype=np.uint8)
            array = array.reshape((image.height, image.width, 4))
            array = array[:, :, :3]
            array = cv2.cvtColor(array, cv2.COLOR_BGR2RGB)
            
            # Apply basic white balance
            array = cv2.cvtColor(array, cv2.COLOR_RGB2LAB)
            avg_a = np.mean(array[:, :, 1])
            avg_b = np.mean(array[:, :, 2])
            array[:, :, 1] = array[:, :, 1] - ((avg_a - 128) * (array[:, :, 0] / 255.0) * 1.1)
            array[:, :, 2] = array[:, :, 2] - ((avg_b - 128) * (array[:, :, 0] / 255.0) * 1.1)
            array = cv2.cvtColor(array, cv2.COLOR_LAB2RGB)
            
            if self.image_queue.empty():
                self.image_queue.put(array)
        except Exception as e:
            print(f"⚠️ Camera processing error: {str(e)}")

    def _on_collision(self, event):
        """Improved collision handling with better classification"""
        try:
            # Classify collision type
            if 'static' in event.other_actor.type_id.lower():
                impulse_threshold = 8000  # Lower threshold for static objects
                collision_type = "STATIC"
            elif 'vehicle' in event.other_actor.type_id.lower():
                impulse_threshold = 5000
                collision_type = "VEHICLE"
            else:
                impulse_threshold = 3000
                collision_type = "OTHER"
                
            if event.normal_impulse.length() > impulse_threshold:
                print(f"💥 Collision with {event.other_actor.type_id} ({collision_type}, Force: {event.normal_impulse.length():.1f})")
                self._initiate_emergency_brake()
        except Exception as e:
            print(f"🔴 Collision handling error: {str(e)}")

    def _on_lane_invasion(self, event):
        """Handle lane invasion events"""
        try:
            lane_types = set([marking.type for marking in event.crossed_lane_markings])
            
            if carla.LaneMarkingType.Solid in lane_types:
                print("⚠️ Crossed solid lane marking!")
                self.lane_departure_warning = True
            elif carla.LaneMarkingType.SolidSolid in lane_types:
                print("🚨 Crossed double solid line!")
                self.lane_departure_warning = True
                
            # Reset warning after short time
            threading.Timer(3.0, self._reset_lane_warning).start()
        except Exception as e:
            print(f"🔴 Lane invasion error: {str(e)}")

    def _reset_lane_warning(self):
        """Reset lane departure warning"""
        self.lane_departure_warning = False

    def _initiate_emergency_brake(self):
        """Improved emergency braking with staged response"""
        if time.time() - self.last_collision_time < 2.0:
            return
            
        self.emergency_brake = True
        self.last_collision_time = time.time()
        
        # Stage 1: Hard brake
        self.vehicle.apply_control(carla.VehicleControl(
            throttle=0.0,
            steer=0.0,
            brake=1.0,
            hand_brake=True
        ))
        print("🛑 Emergency brake activated - Stage 1: Full stop")
        
        # Stage 2: Assess after 1 second
        threading.Timer(1.0, self._assess_collision_recovery).start()

    def _assess_collision_recovery(self):
        """Determine recovery strategy after collision"""
        if not self.running:
            return
            
        # Check if we're stuck (low speed but throttle applied)
        current_speed = self.vehicle.get_velocity().length()
        if current_speed < 0.5 and abs(self.vehicle.get_control().throttle) > 0.1:
            print("🔄 Vehicle appears stuck - initiating recovery")
            self._recover_from_collision()
        else:
            print("↩️ Resuming normal operation")
            self.emergency_brake = False
            if self.current_goal:
                self._calculate_global_route()

    def _recover_from_collision(self):
        """Execute recovery maneuver with better obstacle awareness"""
        if not self.running:
            return
            
        print("🔄 Attempting collision recovery...")
        self.emergency_brake = False
        
        # Check rear clearance before reversing
        rear_clear = True
        vehicle_loc = self.vehicle.get_location()
        vehicle_rot = self.vehicle.get_transform().rotation
        
        # Create rear vector
        rear_vector = carla.Location(
            x=-2.0 * np.cos(np.radians(vehicle_rot.yaw)),
            y=-2.0 * np.sin(np.radians(vehicle_rot.yaw))
        )
        rear_point = vehicle_loc + rear_vector
        
        # Check for obstacles behind
        for actor in self.world.get_actors():
            if actor.id != self.vehicle.id and actor.get_location().distance(rear_point) < 3.0:
                rear_clear = False
                break
        
        if rear_clear:
            # Apply reverse with slight random steering
            self.reverse_mode = True
            self.reverse_start_time = time.time()
            self.vehicle.apply_control(carla.VehicleControl(
                throttle=-0.5,
                steer=random.uniform(-0.3, 0.3),
                brake=0.0,
                reverse=True
            ))
            print("↩️ Reversing to clear obstruction")
            
            # Schedule return to normal operation
            threading.Timer(self.reverse_duration, self._resume_normal_operation).start()
        else:
            print("⚠️ Cannot reverse - obstacle detected behind")
            self._resume_normal_operation()

    def _resume_normal_operation(self):
        """Return to normal driving with route recalculation"""
        if not self.running:
            return
            
        self.reverse_mode = False
        print("↩️ Resuming normal operation")
        if self.current_goal:
            self._calculate_global_route()

    def process_frame(self):
        """Enhanced frame processing with better lane detection"""
        if self.image_queue.empty() or self.vehicle is None:
            return None, []
            
        try:
            frame = self.image_queue.get()
            if frame is None or frame.size == 0:
                return None, []
                
            self.frame_count += 1
            
            # Run detection
            results = self.model(frame, verbose=False)
            if results is None:
                return frame, list(self.tracked_obstacles.values())
            
            # Estimate depth
            depth_map = self._process_depth(frame) if not self.emergency_brake else None
            
            # Process detections with accurate depth
            self._update_tracked_obstacles(results, depth_map)
            
            # Enhanced lane detection when safe
            if not self.emergency_brake and self.lane_keeping_active:
                self._enhanced_lane_detection(frame)
                
            return frame, list(self.tracked_obstacles.values())
            
        except Exception as e:
            print(f"🔴 Frame processing error: {str(e)}")
            return None, []

    def _enhanced_lane_detection(self, frame):
        """Robust lane detection with RANSAC and safety checks"""
        try:
            # Apply perspective transform (bird's eye view)
            warped = cv2.warpPerspective(frame, self.perspective_transform, 
                                        (frame.shape[1], frame.shape[0]))
        
            # Convert to grayscale and apply adaptive threshold
            gray = cv2.cvtColor(warped, cv2.COLOR_RGB2GRAY)
            thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                        cv2.THRESH_BINARY_INV, 11, 2)
        
            # Apply morphological operations
            kernel = np.ones((5,5), np.uint8)
            closed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
        
            # Detect edges
            edges = cv2.Canny(closed, 50, 150)
        
            # Detect lines using probabilistic Hough transform
            lines = cv2.HoughLinesP(edges, 1, np.pi/180, 50, 
                                minLineLength=50, maxLineGap=20)
        
            if lines is not None:
                left_lines = []
                right_lines = []
            
                for line in lines:
                    x1, y1, x2, y2 = line[0]
                
                    # Filter near-vertical lines
                    if abs(x2 - x1) < 10:
                        continue
                    
                    slope = (y2 - y1) / (x2 - x1)
                    intercept = y1 - slope * x1
                
                    # Classify left/right lanes with stricter slope thresholds
                    if slope < -0.5 and x1 < frame.shape[1]//2 and x2 < frame.shape[1]//2:
                        left_lines.append((slope, intercept))
                    elif slope > 0.5 and x1 > frame.shape[1]//2 and x2 > frame.shape[1]//2:
                        right_lines.append((slope, intercept))
            
                # Fit lines using RANSAC with safety checks
                self.lane_boundaries = {
                    'left': self._fit_ransac_line(left_lines, frame.shape[0]),
                    'right': self._fit_ransac_line(right_lines, frame.shape[0])
                }
            
                # Calculate lane center offset only if both lanes detected
                if (self.lane_boundaries['left'] and 
                    self.lane_boundaries['right'] and
                    abs(self.lane_boundaries['left'][0]) > 0.001 and 
                    abs(self.lane_boundaries['right'][0]) > 0.001):
                
                    left_slope, left_intercept = self.lane_boundaries['left']
                    right_slope, right_intercept = self.lane_boundaries['right']
                    y = frame.shape[0]
                
                    # Calculate with safe division
                    left_x = (y - left_intercept) / left_slope if abs(left_slope) > 0.001 else 0
                    right_x = (y - right_intercept) / right_slope if abs(right_slope) > 0.001 else 0
                
                    if left_x and right_x:
                        lane_center = (left_x + right_x) / 2
                        vehicle_center = frame.shape[1] / 2
                        self.lane_offset_history.append(vehicle_center - lane_center)
                    
        except Exception as e:
            print(f"🔴 Enhanced lane detection error: {str(e)}")
            self.lane_boundaries = None

    def _fit_ransac_line(self, lines, img_height):
        """Safe line fitting with RANSAC"""
        if not lines or len(lines) < 2:
            return None
        
        try:
            # Convert lines to points
            points = []
            for slope, intercept in lines:
                x1 = 0
                y1 = int(intercept)
                x2 = img_height
                y2 = int(slope * x2 + intercept)
                points.extend([(x1,y1), (x2,y2)])
        
            # Convert to numpy array
            points = np.array(points)
            X = points[:,0].reshape(-1,1)
            y = points[:,1]
        
            # Filter invalid points
            valid = ~np.isinf(X).any(axis=1) & ~np.isnan(X).any(axis=1)
            X = X[valid]
            y = y[valid]
        
            if len(X) < 2:
                return None
            
            # Fit with RANSAC
            ransac = RANSACRegressor()
            ransac.fit(X, y)
        
            # Get line parameters (slope, intercept)
            slope = ransac.estimator_.coef_[0]
            intercept = ransac.estimator_.intercept_
        
            return (slope, intercept)
        except:
            return None
    def _update_tracked_obstacles(self, results, depth_map):
        """Improved obstacle tracking with Kalman filtering"""
        current_detections = []
        
        try:
            for result in results:
                if not hasattr(result, 'boxes'):
                    continue
                    
                for box in result.boxes:
                    if box.conf.item() < 0.6:
                        continue
                        
                    box_coords = box.xyxy[0].cpu().numpy()
                    x1, y1, x2, y2 = map(float, box_coords)
                    center_x, center_y = (x1 + x2) // 2, (y1 + y2) // 2
                    
                    # Get accurate distance
                    if depth_map is not None:
                        obstacle_distance = self._get_obstacle_distance(depth_map, int(center_x), int(center_y))
                    else:
                        obstacle_distance = 20.0
                    
                    # Estimate width and height
                    width = x2 - x1
                    height = y2 - y1
                    
                    current_detections.append({
                        'type': result.names[int(box.cls.item())],
                        'bbox': [x1, y1, x2, y2],
                        'center': (center_x, center_y),
                        'distance': obstacle_distance,
                        'width': width,
                        'height': height,
                        'timestamp': time.time(),
                        'last_seen': self.frame_count
                    })

            # Improved tracking with simple motion model
            updated_tracks = {}
            for det in current_detections:
                closest_id = None
                min_dist = float('inf')
                
                for track_id, track in self.tracked_obstacles.items():
                    try:
                        # Calculate IoU and center distance
                        box_a = track['bbox']
                        box_b = det['bbox']
                        
                        # Calculate intersection area
                        xA = max(box_a[0], box_b[0])
                        yA = max(box_a[1], box_b[1])
                        xB = min(box_a[2], box_b[2])
                        yB = min(box_a[3], box_b[3])
                        
                        inter_area = max(0, xB - xA) * max(0, yB - yA)
                        
                        # Calculate union area
                        box_a_area = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
                        box_b_area = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
                        union_area = box_a_area + box_b_area - inter_area
                        
                        iou = inter_area / union_area if union_area > 0 else 0
                        
                        # Calculate center distance
                        center_dist = np.sqrt((det['center'][0] - track['center'][0])**2 + 
                                            (det['center'][1] - track['center'][1])**2)
                        
                        # Combined score (prioritize IoU)
                        score = 0.7 * (1 - iou) + 0.3 * (center_dist / 100)
                        
                        if score < min_dist and iou > 0.3:
                            closest_id = track_id
                            min_dist = score
                    except:
                        continue
                
                if closest_id is not None:
                    # Update with simple motion model
                    updated_tracks[closest_id] = {
                        **self.tracked_obstacles[closest_id],
                        **det,
                        'distance': 0.7 * self.tracked_obstacles[closest_id]['distance'] + 0.3 * det['distance'],
                        'bbox': det['bbox'],  # Update with new bbox
                        'velocity': self._estimate_velocity(self.tracked_obstacles[closest_id], det)
                    }
                else:
                    new_id = max(self.tracked_obstacles.keys(), default=0) + 1
                    updated_tracks[new_id] = {**det, 'velocity': 0.0}
            
            self.tracked_obstacles = {
                k: v for k, v in updated_tracks.items() 
                if self.frame_count - v['last_seen'] < 5
            }
            
        except Exception as e:
            print(f"🔴 Major tracking error: {str(e)}")
            self.tracked_obstacles = {k: v for k, v in self.tracked_obstacles.items() 
                                    if self.frame_count - v['last_seen'] < 2}

    def _estimate_velocity(self, prev_det, current_det):
        """Simple velocity estimation between frames"""
        try:
            if 'center' not in prev_det or 'center' not in current_det:
                return 0.0
                
            # Calculate pixel displacement
            dx = current_det['center'][0] - prev_det['center'][0]
            dy = current_det['center'][1] - prev_det['center'][1]
            
            # Time difference in seconds
            dt = current_det['timestamp'] - prev_det['timestamp']
            if dt <= 0:
                return 0.0
                
            # Convert to m/s (very rough approximation)
            # Assuming camera FOV of 90° and distance to object
            avg_distance = (prev_det['distance'] + current_det['distance']) / 2
            if avg_distance <= 0:
                return 0.0
                
            # Approximate conversion factor (pixels to meters)
            # This is a simplified calculation - should be calibrated
            fov_rad = np.radians(90)
            px_per_rad = frame.shape[1] / fov_rad
            meters_per_px = avg_distance * np.tan(1/px_per_rad)
            
            velocity = np.sqrt(dx**2 + dy**2) * meters_per_px / dt
            return velocity
        except:
            return 0.0

    def set_destination(self, location):
        """Improved destination setting with route validation"""
        waypoint = self.map.get_waypoint(location)
        if waypoint and waypoint.lane_type == carla.LaneType.Driving:
            self.current_goal = waypoint
            self._calculate_global_route()
            print(f"📍 Destination set to {location}")
            return True
        else:
            print(f"⚠️ Invalid destination {location} - not on drivable road")
            return False
        
    def _calculate_global_route(self):
        """Fixed route calculation with proper waypoint access"""
        if not self.current_goal:
            return
        
        try:
            start = self.map.get_waypoint(self.vehicle.get_location())
            grp = self.map.get_waypoint(start.transform.location).next_until_lane_end(2.0)
        
            if not grp:
                print("⚠️ Could not find starting waypoint for route planning")
                return
            
            self.global_route = []
            current = start
        
            while current and len(self.global_route) < 100:
                next_waypoints = current.next(5.0)
                if not next_waypoints:
                    break
                
                next_wp = min(next_waypoints,
                            key=lambda wp: wp.transform.location.distance(self.current_goal.transform.location))
            
                self.global_route.append(next_wp)
                current = next_wp
            
                if current.transform.location.distance(self.current_goal.transform.location) < 10.0:
                    break
                
            print(f"🛣️ Calculated route with {len(self.global_route)} waypoints")
        
        except Exception as e:
            print(f"🔴 Route calculation error: {str(e)}")
            self.global_route = []

    def _find_closest_waypoint_index(self, location):
        """Find index of closest waypoint in global route"""
        if not self.global_route:
            return 0
            
        distances = [wp.transform.location.distance(location) for wp in self.global_route]
        return np.argmin(distances)

    def _detect_static_obstacles(self, waypoint):
        """Enhanced static obstacle detection with semantic segmentation"""
        vehicle_loc = self.vehicle.get_location()
        obstacles = []
        
        # Get all static objects in the world
        for actor in self.world.get_actors():
            if 'static' in actor.type_id.lower():
                loc = actor.get_location()
                distance = loc.distance(vehicle_loc)
                
                if distance < self.static_obstacle_range:
                    # Calculate angle from vehicle's forward vector
                    forward = self.vehicle.get_transform().get_forward_vector()
                    to_obstacle = loc - vehicle_loc
                    angle = np.degrees(np.arccos(
                        forward.dot(to_obstacle) / (forward.length() * to_obstacle.length())
                    ))
                    
                    # Consider obstacles in a 60° cone in front
                    if angle < 60.0:
                        # Get waypoint at obstacle location
                        obs_wp = self.map.get_waypoint(loc)
                        if obs_wp and obs_wp.lane_id == waypoint.lane_id:
                            obstacles.append({
                                'location': loc,
                                'distance': distance,
                                'type': actor.type_id,
                                'waypoint': obs_wp
                            })
        
        return obstacles

    def plan_path(self):
        """Enhanced obstacle-aware path planning"""
        if not self.global_route or self.reverse_mode:
            return None
            
        vehicle_loc = self.vehicle.get_location()
        current_idx = self._find_closest_waypoint_index(vehicle_loc)
        
        # Look ahead 5 waypoints (about 25m)
        lookahead = 5
        target_idx = min(current_idx + lookahead, len(self.global_route) - 1)
        target_waypoint = self.global_route[target_idx]
        
        # Check for static obstacles
        static_obstacles = self._detect_static_obstacles(target_waypoint)
        if static_obstacles:
            print(f"⚠️ Detected {len(static_obstacles)} static obstacles")
            return self._generate_avoidance_path(target_waypoint, static_obstacles)
        
        # Check for dynamic obstacles
        dynamic_obstacles = [obs for obs in self.tracked_obstacles.values() 
                            if obs['distance'] < 15 and obs['type'] in ['car', 'truck', 'bus']]
        if dynamic_obstacles:
            return self._handle_dynamic_obstacles(target_waypoint, dynamic_obstacles)
        
        return target_waypoint

    def _generate_avoidance_path(self, original_waypoint, obstacles):
        """Generate alternative path considering obstacles"""
        # Try left lane first
        left_wp = original_waypoint.get_left_lane()
        if left_wp and left_wp.lane_type == carla.LaneType.Driving:
            left_clear = True
            for obs in obstacles:
                if obs['waypoint'].lane_id == left_wp.lane_id:
                    left_clear = False
                    break
            if left_clear:
                print("🔄 Taking left lane to avoid obstacle")
                return left_wp
        
        # Try right lane if left is blocked
        right_wp = original_waypoint.get_right_lane()
        if right_wp and right_wp.lane_type == carla.LaneType.Driving:
            right_clear = True
            for obs in obstacles:
                if obs['waypoint'].lane_id == right_wp.lane_id:
                    right_clear = False
                    break
            if right_clear:
                print("🔄 Taking right lane to avoid obstacle")
                return right_wp
        
        # If no lane changes possible, find closest passable point
        if obstacles:
            closest_obs = min(obstacles, key=lambda x: x['distance'])
            if closest_obs['distance'] > 5.0:  # If we have some space
                # Try to shift within current lane
                shifted_wp = original_waypoint
                if closest_obs['location'].x > shifted_wp.transform.location.x:
                    shifted_wp = shifted_wp.next(2.0)[0].get_left_lane()
                else:
                    shifted_wp = shifted_wp.next(2.0)[0].get_right_lane()
                
                if shifted_wp and shifted_wp.lane_type == carla.LaneType.Driving:
                    print("🔄 Shifting position to avoid obstacle")
                    return shifted_wp
        
        # Last resort - stop and wait
        print("⚠️ No clear path found - stopping")
        return None

    def _handle_dynamic_obstacles(self, target_waypoint, obstacles):
        """Handle moving obstacles with velocity prediction"""
        vehicle_loc = self.vehicle.get_location()
        vehicle_speed = self.vehicle.get_velocity().length()
        
        # Predict obstacle positions in 1 second
        dangerous_obstacles = []
        for obs in obstacles:
            predicted_distance = obs['distance'] - obs['velocity'] * 1.0
            if predicted_distance < 8.0:  # Dangerous within next second
                dangerous_obstacles.append(obs)
        
        if not dangerous_obstacles:
            return target_waypoint
            
        # Find the most threatening obstacle
        closest_obs = min(dangerous_obstacles, key=lambda x: x['distance'])
        
        # Decide action based on relative speed
        if closest_obs['velocity'] < vehicle_speed + 2.0:  # Approaching slowly
            print("⚠️ Approaching slower vehicle - maintaining distance")
            return self.global_route[max(0, self._find_closest_waypoint_index(vehicle_loc) - 2)]
        else:
            # Consider lane change if possible
            left_wp = target_waypoint.get_left_lane()
            right_wp = target_waypoint.get_right_lane()
            
            # Prefer lane opposite to obstacle's relative position
            obs_rel_x = closest_obs['center'][0] - 640  # Relative to image center
            if abs(obs_rel_x) > 100:  # Significant lateral offset
                if obs_rel_x > 0 and left_wp:  # Obstacle is right, move left
                    print("🔄 Changing left to avoid oncoming vehicle")
                    return left_wp
                elif obs_rel_x < 0 and right_wp:  # Obstacle is left, move right
                    print("🔄 Changing right to avoid oncoming vehicle")
                    return right_wp
            
            # Default to slowing down
            print("⚠️ Oncoming vehicle - slowing down")
            return self.global_route[max(0, self._find_closest_waypoint_index(vehicle_loc) - 3)]

    def calculate_steering(self, target_waypoint):
        """Enhanced steering control with lane centering"""
        if not target_waypoint:
            return 0.0
            
        # Get vehicle state
        vehicle_transform = self.vehicle.get_transform()
        vehicle_loc = vehicle_transform.location
        vehicle_yaw = np.radians(vehicle_transform.rotation.yaw)
        current_speed = self.vehicle.get_velocity().length()
        
        # Lane keeping component (weight increases with speed)
        lane_steer = 0.0
        if self.lane_boundaries and len(self.lane_offset_history) > 0:
            avg_offset = np.mean(self.lane_offset_history)
            lane_weight = min(1.0, current_speed / 10.0)  # More weight at higher speeds
            lane_steer = -avg_offset * 0.01 * lane_weight
            
            # Add lane departure correction
            if self.lane_departure_warning:
                lane_steer *= 1.5  # More aggressive correction
        
        # Waypoint following component
        target_vec = np.array([
            target_waypoint.transform.location.x - vehicle_loc.x,
            target_waypoint.transform.location.y - vehicle_loc.y
        ])
        forward_vec = np.array([np.cos(vehicle_yaw), np.sin(vehicle_yaw)])
        cte = np.cross(forward_vec, target_vec/np.linalg.norm(target_vec))
        
        dt = time.time() - self.last_control_time
        wp_steer = self.pid_steer.update(cte, dt)
        
        # Blend steering based on conditions
        if self.lane_change_in_progress:
            # Prioritize waypoint following during lane changes
            blend_ratio = 0.3  # 30% lane keeping, 70% waypoint
        else:
            # Normal blending (more lane keeping at high speed)
            blend_ratio = 0.7 - (0.3 * min(1.0, current_speed / 20.0))
        
        final_steer = blend_ratio * lane_steer + (1 - blend_ratio) * wp_steer
        
        # Add curvature anticipation
        if len(self.global_route) > 3:
            current_idx = self._find_closest_waypoint_index(vehicle_loc)
            if current_idx < len(self.global_route) - 3:
                next_wp1 = self.global_route[current_idx + 1]
                next_wp2 = self.global_route[current_idx + 2]
                
                # Calculate curvature
                angle1 = np.arctan2(next_wp1.transform.location.y - vehicle_loc.y,
                                  next_wp1.transform.location.x - vehicle_loc.x)
                angle2 = np.arctan2(next_wp2.transform.location.y - next_wp1.transform.location.y,
                                  next_wp2.transform.location.x - next_wp1.transform.location.x)
                curve = angle2 - angle1
                
                # Add anticipatory steering
                final_steer += 0.5 * curve * min(1.0, current_speed / 15.0)
        
        # Smooth steering using history
        self.steering_history.append(final_steer)
        if len(self.steering_history) == self.steering_history.maxlen:
            final_steer = np.mean(self.steering_history)
        
        self.last_control_time = time.time()
        return np.clip(final_steer, -1.0, 1.0)

    def calculate_throttle(self):
        """Enhanced speed control with obstacle awareness"""
        current_speed = self.vehicle.get_velocity().length() * 3.6  # km/h
        speed_error = self.target_speed - current_speed
        dt = time.time() - self.last_control_time
        base_throttle = self.pid_speed.update(speed_error, dt)
        
        # Reduce speed based on closest obstacle
        closest_obstacle = min(self.tracked_obstacles.values(), 
                              key=lambda x: x['distance'], default=None)
        if closest_obstacle:
            # Dynamic safety distance based on speed
            safe_distance = max(5.0, current_speed / 5.0)
            if closest_obstacle['distance'] < safe_distance:
                # Reduce throttle proportionally
                danger_ratio = min(1.0, (safe_distance - closest_obstacle['distance']) / safe_distance)
                base_throttle *= (1.0 - danger_ratio)
                
                # Apply gentle braking if too close
                if closest_obstacle['distance'] < safe_distance * 0.5:
                    base_throttle = min(base_throttle, 0.0)
        
        # Reduce speed in curves
        if abs(self.vehicle.get_control().steer) > 0.2:
            curve_factor = 1.0 - (abs(self.vehicle.get_control().steer) * 0.8)
            base_throttle *= curve_factor
            
        # Gradual acceleration from stop
        if current_speed < 5.0 and base_throttle > 0:
            base_throttle = min(base_throttle, 0.3)
        
        return np.clip(base_throttle, 0.0, 1.0)
    def _initiate_reverse_maneuver(self):
        """Initiate a controlled reverse maneuver to avoid obstacles"""
        if time.time() - self.last_recovery_time < self.recovery_cooldown:
            return
    
        print("🔄 Initiating reverse maneuver")
        self.reverse_mode = True
        self.reverse_start_time = time.time()
        self.last_recovery_time = time.time()
    
        # Apply reverse with slight steering
        self.vehicle.apply_control(carla.VehicleControl(
            throttle=-0.5,
            steer=random.uniform(-0.2, 0.2),  # Small random steering
            brake=0.0,
            reverse=True
        ))
    
        # Schedule return to normal operation
        threading.Timer(self.reverse_duration, self._resume_normal_operation).start()

    def apply_control(self):
        """Enhanced vehicle control with comprehensive safety checks"""
        # Emergency brake handling (highest priority)
        if self.emergency_brake:
            if time.time() - self.last_collision_time > self.collision_cooldown:
                self.emergency_brake = False
            else:
                self.vehicle.apply_control(carla.VehicleControl(
                    throttle=0.0, 
                    steer=0.0, 
                    brake=1.0,
                    hand_brake=True
                ))
                return

        # Get closest obstacle information
        closest_obstacle = min(self.tracked_obstacles.values(), 
                            key=lambda x: x['distance'], default=None)
        current_speed = self.vehicle.get_velocity().length() * 3.6  # km/h

        # Reverse maneuver logic
        if not self.reverse_mode and time.time() - self.last_recovery_time > self.recovery_cooldown:
            if closest_obstacle and closest_obstacle['distance'] < self.min_reverse_distance:
                # Check if we're actually stuck (not just passing by an obstacle)
                if current_speed < 5.0:  # Only reverse if we're moving very slowly
                    self._initiate_reverse_maneuver()
    
        # Handle active reverse maneuver
        if self.reverse_mode:
            if time.time() - self.reverse_start_time < self.reverse_duration:
                # Apply reverse with slight steering variation to help get unstuck
                self.vehicle.apply_control(carla.VehicleControl(
                    throttle=-0.5,
                    steer=random.uniform(-0.3, 0.3),
                    brake=0.0,
                    reverse=True
                ))
                return
            else:
                print("🔄 Re-planning route after reverse maneuver")
                self.reverse_mode = False
                if self.current_goal:
                    self._calculate_global_route()

        # Normal driving control
        target_waypoint = self.plan_path()
        if not target_waypoint:
            # No valid path found - stop and wait
            self.vehicle.apply_control(carla.VehicleControl(
                throttle=0.0,
                steer=0.0,
                brake=1.0,
                hand_brake=True
            ))
            return

        # Calculate base control values
        throttle = self.calculate_throttle()
        steer = self.calculate_steering(target_waypoint)
        brake = 0.0

        # Dynamic braking based on obstacles
        if closest_obstacle:
            # Calculate safe distance based on current speed (2 second rule)
            safe_distance = max(3.0, current_speed / 2.0)
        
            # Emergency brake if collision is imminent
            if closest_obstacle['distance'] < 2.0:
                brake = 1.0
                throttle = 0.0
            # Progressive braking as we approach obstacles
            elif closest_obstacle['distance'] < safe_distance:
                brake = min(0.7, (safe_distance - closest_obstacle['distance']) / safe_distance)
                throttle *= 0.5  # Reduce throttle when braking

        # Smooth control transitions
        current_control = self.vehicle.get_control()
        smooth_throttle = 0.6 * current_control.throttle + 0.4 * throttle
        smooth_steer = 0.6 * current_control.steer + 0.4 * steer
        smooth_brake = 0.6 * current_control.brake + 0.4 * brake

        # Apply final control values with clamping
        self.vehicle.apply_control(carla.VehicleControl(
            throttle=float(np.clip(smooth_throttle, 0.0, 1.0)),
            steer=float(np.clip(smooth_steer, -1.0, 1.0)),
            brake=float(np.clip(smooth_brake, 0.0, 1.0)),
            reverse=False
        ))

        # Update waypoint and log data
        self.current_waypoint = target_waypoint
        self._log_data(throttle, steer, brake) 
    def _is_lane_clear(self, waypoint, lookahead=20.0):
        """Check if a lane is clear of obstacles for a given distance"""
        current_loc = self.vehicle.get_location()
        for i in range(int(lookahead/5)):
            next_wps = waypoint.next(5.0)
            if not next_wps:
                return True
            waypoint = next_wps[0]
        
            # Check for obstacles in this segment
            for obs in self.tracked_obstacles.values():
                if obs['location'].distance(waypoint.transform.location) < 5.0:
                    return False
        return True

    def _complete_lane_change(self):
        """Complete an active lane change maneuver"""
        current_loc = self.vehicle.get_location()
        target_loc = self.lane_change_target.transform.location
    
        # Check if we've reached the target lane
        if current_loc.distance(target_loc) < 3.0:
            print("✅ Lane change completed")
            return True
    
        # Continue steering toward target lane
        target_vec = np.array([target_loc.x - current_loc.x, 
                            target_loc.y - current_loc.y])
        forward_vec = self.vehicle.get_transform().get_forward_vector()
        forward_vec = np.array([forward_vec.x, forward_vec.y])
    
        cte = np.cross(forward_vec, target_vec/np.linalg.norm(target_vec))
        steer = np.clip(cte * 2.0, -0.5, 0.5)  # Gentle steering
    
        self.vehicle.apply_control(carla.VehicleControl(
            throttle=0.5,
            steer=float(steer),
            brake=0.0
        ))
        return False

    def _calculate_dynamic_controls(self, target_waypoint, closest_obstacle):
        """Calculate throttle, steer and brake with enhanced obstacle response"""
        throttle = self.calculate_throttle()
        steer = self.calculate_steering(target_waypoint)
        brake = 0.0
    
        if closest_obstacle:
            # Dynamic safety distance based on speed (3 second rule)
            safe_distance = max(5.0, self.vehicle.get_velocity().length() * 3.0)
            actual_distance = closest_obstacle['distance']
        
            if actual_distance < safe_distance * 0.3:  # Emergency stop
                brake = 1.0
                throttle = 0.0
            elif actual_distance < safe_distance:
                # Progressive braking with distance ratio
                brake_ratio = (safe_distance - actual_distance) / safe_distance
                brake = min(0.7, brake_ratio)
                throttle *= (1.0 - brake_ratio)  # Reduce throttle proportionally
            
                # If following for too long, consider lane change again
                if brake_ratio > 0.5 and time.time() - self.last_obstacle_detected > 5.0:
                    self.last_obstacle_detected = time.time()
    
        return throttle, steer, brake

    def _apply_smooth_controls(self, throttle, steer, brake):
        """Apply controls with smooth transitions and safety checks"""
        current_control = self.vehicle.get_control()
    
        # Use different smoothing factors for different situations
        if abs(current_control.steer - steer) > 0.3:  # Large steering change
            smooth_factor = 0.3  # Faster response
        else:
            smooth_factor = 0.7  # Smoother normal operation
    
        smooth_throttle = (smooth_factor * current_control.throttle + 
                        (1-smooth_factor) * throttle)
        smooth_steer = (smooth_factor * current_control.steer + 
                    (1-smooth_factor) * steer)
        smooth_brake = (smooth_factor * current_control.brake + 
                    (1-smooth_factor) * brake)
    
        # Apply final control values with clamping
        self.vehicle.apply_control(carla.VehicleControl(
            throttle=float(np.clip(smooth_throttle, 0.0, 1.0)),
            steer=float(np.clip(smooth_steer, -1.0, 1.0)),
            brake=float(np.clip(smooth_brake, 0.0, 1.0)),
            reverse=False
        ))  

    def _safety_monitor(self):
        """Enhanced safety monitoring with multiple checks"""
        while self.running:
            try:
                if not self.vehicle:
                    time.sleep(0.1)
                    continue
                    
                # Check road boundaries
                if not self._check_on_road():
                    self.off_road_counter += 1
                    if self.off_road_counter > self.max_off_road_frames:
                        print("🚨 Vehicle off-road detected!")
                        self.safety_status = "OFF_ROAD"
                        self._initiate_emergency_brake()
                        self._recover_to_road()
                else:
                    self.off_road_counter = 0
                    self.safety_status = "NORMAL"
                
                # Collision imminent check
                if len(self.tracked_obstacles) > 0:
                    closest = min(self.tracked_obstacles.values(), 
                                key=lambda x: x['distance'])
                    
                    # Dynamic threshold based on speed
                    speed = self.vehicle.get_velocity().length()
                    threshold = max(2.0, speed * 0.5)  # At least 2m, or 0.5s at current speed
                    
                    if closest['distance'] < threshold:
                        self.safety_status = "COLLISION_IMMINENT"
                        if closest['distance'] < threshold * 0.7:
                            self._initiate_emergency_brake()
                
                # Lane departure warning
                if (self.lane_boundaries and 
                    len(self.lane_offset_history) > 0 and 
                    abs(np.mean(self.lane_offset_history)) > self.lane_change_threshold):
                    self.lane_departure_warning = True
                else:
                    self.lane_departure_warning = False
                
                time.sleep(0.1)
            except Exception as e:
                print(f"⚠️ Safety monitor error: {str(e)}")
                time.sleep(1)

    def _recover_to_road(self):
        """Improved road recovery with path replanning"""
        nearest_waypoint = self.map.get_waypoint(self.vehicle.get_location())
        if nearest_waypoint:
            # Calculate recovery vector
            recovery_vector = nearest_waypoint.transform.location - self.vehicle.get_location()
            recovery_yaw = np.arctan2(recovery_vector.y, recovery_vector.x)
            current_yaw = np.radians(self.vehicle.get_transform().rotation.yaw)
            angle_diff = recovery_yaw - current_yaw
            
            # Normalize angle difference
            angle_diff = (angle_diff + np.pi) % (2 * np.pi) - np.pi
            
            # Apply control
            self.vehicle.apply_control(carla.VehicleControl(
                throttle=0.3,
                steer=np.clip(angle_diff * 1.5, -1.0, 1.0),
                brake=0.0
            ))
            
            # Replan route after recovery
            threading.Timer(2.0, self._calculate_global_route).start()

    def _check_on_road(self):
        """Strict road surface verification to avoid footpaths and off-road driving"""
        try:
            waypoint = self.map.get_waypoint(self.vehicle.get_location(), project_to_road=False)
            if not waypoint:
                return False

            # Check if the vehicle is strictly on a Driving lane
            if waypoint.lane_type != carla.LaneType.Driving:
                return False

            if hasattr(waypoint, 'lane_change') and waypoint.lane_change != carla.LaneChange.NONE:
                return False


            # Check distance from both sidewalks
            vehicle_loc = self.vehicle.get_location()
            left = waypoint.get_left_lane()
            right = waypoint.get_right_lane()

            for side in [left, right]:
                if side and side.lane_type == carla.LaneType.Sidewalk:
                    dist = vehicle_loc.distance(side.transform.location)
                    if dist < 1.5:  # Too close to sidewalk
                        return False

            return True
        except Exception as e:
            print(f"🔴 Enhanced road check error: {str(e)}")
            return False

    def _create_depth_visualization(self, depth_map):
        """Create depth visualization with proper normalization"""
        if depth_map is None:
            return None
    
        try:
            # Normalize and apply color map
            depth_vis = cv2.normalize(depth_map, None, 0, 255, cv2.NORM_MINMAX)
            depth_vis = cv2.applyColorMap(depth_vis.astype(np.uint8), cv2.COLORMAP_JET)
        
            # Add distance scale
            cv2.putText(depth_vis, "NEAR", (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
            cv2.putText(depth_vis, "FAR", (depth_vis.shape[1]-50, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
        
            # Add dynamic distance indicator
            if self.tracked_obstacles:
                closest = min(self.tracked_obstacles.values(), 
                            key=lambda x: x['distance'])
                cv2.putText(depth_vis, f"{closest['distance']:.1f}m", 
                        (depth_vis.shape[1]//2, depth_vis.shape[0]-20), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, 
                        (0,0,255) if closest['distance'] < 5 else (255,255,255), 2)
        
            return cv2.resize(depth_vis, (640, 360))
        except Exception as e:
            print(f"⚠️ Depth visualization error: {str(e)}")
            return None

    def visualize(self, frame, obstacles):
        """Robust visualization system with error handling"""
        if frame is None or frame.size == 0:
            return None
        
        try:
            # Create display frame
            display_frame = cv2.resize(frame, (1280, 720))
        
            # Draw lane boundaries if detected
            if (self.lane_boundaries and 
                self.lane_boundaries['left'] and 
                self.lane_boundaries['right']):
            
                left_slope, left_intercept = self.lane_boundaries['left']
                right_slope, right_intercept = self.lane_boundaries['right']
                y1, y2 = 0, display_frame.shape[0]
            
                # Calculate points with safe division
                left_x1 = (y1 - left_intercept)/left_slope if abs(left_slope) > 0.001 else 0
                left_x2 = (y2 - left_intercept)/left_slope if abs(left_slope) > 0.001 else 0
                right_x1 = (y1 - right_intercept)/right_slope if abs(right_slope) > 0.001 else 0
                right_x2 = (y2 - right_intercept)/right_slope if abs(right_slope) > 0.001 else 0
            
                # Draw left lane (yellow)
                if left_x1 and left_x2:
                    cv2.line(display_frame, 
                            (int(left_x1), y1), (int(left_x2), y2),
                            (0, 255, 255), 2)
            
                # Draw right lane (yellow)
                if right_x1 and right_x2:
                    cv2.line(display_frame,
                            (int(right_x1), y1), (int(right_x2), y2),
                            (0, 255, 255), 2)
        
            # Draw detections
            for obstacle in obstacles:
                try:
                    x1, y1, x2, y2 = map(int, obstacle['bbox'])
                
                    # Color coding by distance
                    if obstacle['distance'] < 5:
                        color = (0, 0, 255)    # Red
                    elif obstacle['distance'] < 10:
                        color = (0, 165, 255)  # Orange
                    else:
                        color = (0, 255, 0)    # Green
                
                    # Draw bounding box
                    cv2.rectangle(display_frame, (x1, y1), (x2, y2), color, 2)
                
                    # Draw info text
                    info = f"{obstacle['type']} {obstacle['distance']:.1f}m"
                    if 'velocity' in obstacle:
                        info += f" {obstacle['velocity']:.1f}m/s"
                
                    cv2.putText(display_frame, info,
                            (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                            (255,255,255), 1)
                except:
                    continue
        
            # Draw route if available
            if self.global_route and len(self.global_route) > 1:
                for i in range(len(self.global_route)-1):
                    try:
                        p1 = self._world_to_pixel(self.global_route[i].transform.location)
                        p2 = self._world_to_pixel(self.global_route[i+1].transform.location)
                        if p1 and p2:
                            cv2.line(display_frame, p1, p2, (255,0,0), 2)
                    except:
                        continue
        
            # Add status overlay
            overlay = self._create_parameter_display()
        
            # Combine and display
            combined = np.vstack([display_frame, overlay])
            cv2.imshow('Autonomous Driving', combined)
        
            # Show depth map if available
            if not self.image_queue.empty():
                try:
                    depth_frame = self.image_queue.get()
                    depth_map = self._process_depth(depth_frame)
                    if depth_map is not None:
                        depth_vis = self._create_depth_visualization(depth_map)
                        if depth_vis is not None:
                            cv2.imshow('Depth Estimation', depth_vis)  # Fixed typo here (changed v2 to cv2)
                except:
                    pass
        
            return combined
        
        except Exception as e:
            print(f"🔴 Visualization error: {str(e)}")
            return None
    def _create_parameter_display(self):
        """Enhanced parameter display with more information"""
        overlay = np.zeros((180, 1280, 3), dtype=np.uint8)
        
        # Section headers
        headers = [("VEHICLE STATUS", 20), ("CONTROL INPUTS", 450), ("NAVIGATION", 850)]
        for text, x in headers:
            cv2.putText(overlay, text, (x, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        
        # Vehicle status
        speed = self.vehicle.get_velocity().length() * 3.6
        cv2.putText(overlay, f"Speed: {speed:.1f} km/h", (20, 70), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        road_status = "ON ROAD" if self._check_on_road() else "OFF ROAD"
        road_color = (0, 255, 0) if road_status == "ON ROAD" else (0, 0, 255)
        cv2.putText(overlay, f"Status: {road_status}", (20, 110), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, road_color, 2)
        
        # Control inputs
        control = self.vehicle.get_control()
        cv2.putText(overlay, f"Throttle: {control.throttle:.2f}", (450, 70), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(overlay, f"Steering: {control.steer:.2f}", (450, 110), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        # Navigation info
        cv2.putText(overlay, f"Obstacles: {len(self.tracked_obstacles)}", (850, 70), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        if self.current_goal:
            distance = self.vehicle.get_location().distance(self.current_goal.transform.location)
            cv2.putText(overlay, f"Goal: {distance:.1f}m", (850, 110), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        # Safety status
        cv2.putText(overlay, f"Safety: {self.safety_status}", (20, 150), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, 
                   (0, 0, 255) if self.safety_status != "NORMAL" else (0, 255, 0), 2)
        
        # Lane departure warning
        if self.lane_departure_warning:
            cv2.putText(overlay, "LANE DEPARTURE!", (450, 150), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        
        return overlay

    def _log_data(self, throttle, steer, brake):
        """Enhanced data logging with more parameters"""
        current_speed = self.vehicle.get_velocity().length() * 3.6
        road_status = "ON_ROAD" if self._check_on_road() else "OFF_ROAD"
        collision_status = "EMERGENCY" if self.emergency_brake else "CLEAR"
        lane_offset = np.mean(self.lane_offset_history) if self.lane_offset_history else 0.0
        closest_obstacle = min(self.tracked_obstacles.values(), 
                             key=lambda x: x['distance'], default={'type': 'NONE', 'distance': 0})
        
        log_entry = [
            datetime.now().isoformat(),
            f"{current_speed:.1f}",
            f"{throttle:.2f}",
            f"{steer:.2f}",
            f"{brake:.2f}",
            str(len(self.tracked_obstacles)),
            road_status,
            collision_status,
            f"{lane_offset:.1f}",
            closest_obstacle['type'],
            f"{closest_obstacle['distance']:.1f}",
            self.safety_status,
            "1" if self.lane_departure_warning else "0",
            "1" if self.reverse_mode else "0"
        ]
        self.log_file.write(','.join(log_entry) + '\n')
        self.log_file.flush()

    def run(self):
        """Main execution loop with enhanced setup"""
        if not self.spawn_vehicle():
            print("🔴 Failed to spawn vehicle")
            return False
        
        if not self.setup_sensors():
            print("🔴 Failed to initialize sensors")
            return False
        
        # Set random destination away from current location
        spawn_points = [sp for sp in self.map.get_spawn_points() 
                       if sp.location.distance(self.vehicle.get_location()) > 50.0]
        if spawn_points:
            destination = random.choice(spawn_points).location
            if not self.set_destination(destination):
                print("🔴 Failed to set valid destination")
                return False
        
        self.running = True
        self.safety_thread.start()
        print("🟢 Starting autonomous driving")
        
        try:
            while self.running:
                self.world.tick()
                
                # Process sensor data
                frame, obstacles = self.process_frame()
                
                # Emergency handling
                if self.emergency_brake:
                    if time.time() - self.last_collision_time > self.collision_cooldown:
                        self.emergency_brake = False
                    else:
                        self.apply_control()
                        continue
                
                # Navigation and control
                self.apply_control()
                
                # Visualization
                if frame is not None:
                    _ = self.visualize(frame, obstacles)
                    
                # Check for exit command
                if cv2.waitKey(1) == ord('q'):
                    self.running = False
                    
        except KeyboardInterrupt:
            print("\n🛑 Simulation stopped by user")
        except Exception as e:
            print(f"🔴 Critical error: {str(e)}")
        finally:
            self.cleanup()
            return True

if __name__ == "__main__":
    # Configure environment for Jupyter if needed
    if 'ipykernel' in sys.modules:
        os.environ['DISPLAY'] = ':0'
    
    # Run simulation
    av = AutonomousVehicle()
    if av.run():
        print("🟢 Simulation completed successfully")
    else:
        print("🔴 Simulation failed")

🟢 Perception system initialized
Loading weights:  None


Using cache found in /tmp/torch_hub\rwightman_gen-efficientnet-pytorch_master


✅ Depth estimation initialized
🚗 Spawned vehicle.harley-davidson.low_rider at safe location
📷 Camera sensor ready
⚠️ Collision sensor ready
🛣️ Lane invasion sensor ready
🛣️ Calculated route with 100 waypoints
📍 Destination set to Location(x=106.002838, y=92.812851, z=0.600000)
🟢 Starting autonomous driving
🚨 Vehicle off-road detected!
🛑 Emergency brake activated - Stage 1: Full stop
🚨 Vehicle off-road detected!
🚨 Vehicle off-road detected!
🚨 Vehicle off-road detected!
🚨 Vehicle off-road detected!
🚨 Vehicle off-road detected!
🚨 Vehicle off-road detected!
🚨 Vehicle off-road detected!
🚨 Vehicle off-road detected!
🚨 Vehicle off-road detected!
↩️ Resuming normal operation
🛣️ Calculated route with 100 waypoints
🚨 Vehicle off-road detected!
🚨 Vehicle off-road detected!
🚨 Vehicle off-road detected!
🚨 Vehicle off-road detected!
🚨 Vehicle off-road detected!
🚨 Vehicle off-road detected!
🚨 Vehicle off-road detected!
🚨 Vehicle off-road detected!
🛣️ Calculated route with 100 waypoints
🚨 Vehicle off-